In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## Paramètres du Portefeuille

Nous gérons un portefeuille composé de deux actifs (A et B) dont les rendements journaliers suivent des lois normales mais sont corrélés.

| Actif | Rendement Moyen ($\mu$) | Volatilité ($\sigma$) | Poids ($w$) |
| :---: | :---------------------: | :--------------------: | :---------: |
| **A** | $0.001$                 | $0.02$                 | $0.5$       |
| **B** | $0.0005$                | $0.01$                 | $0.5$       |

**Corrélation ($\rho$) :** $0.6$

---

In [2]:
mu_A = 0.001     
sigma_A = 0.02    
mu_B = 0.0005    
sigma_B = 0.01   
rho = 0.6
w_A = 0.5
w_B = 0.5
N = 10000


**Question 1 : Génération des Scénarios Corréles**

En utilisant la méthode de Cholesky (similaire à l'Exercice 2), on genere  $N = 10000$ scénarios de rendements journaliers conjoints $(R_A, R_B)$.

*   **Taille de la simulation :** $N = 10000$
*   **Méthode :** Décomposition de Cholesky de la matrice de covariance.

In [13]:
cov_AB = rho * sigma_A * sigma_B
Sigma = np.array([[sigma_A**2, cov_AB],[cov_AB, sigma_B**2]])
L = np.linalg.cholesky(Sigma)
np.random.seed(42)  
Z = np.random.randn(2, N)  
R = np.zeros((2, N))
R[0] = mu_A + L[0, 0] * Z[0] + L[0, 1] * Z[1]
R[1] = mu_B + L[1, 0] * Z[0] + L[1, 1] * Z[1]

**Question 2 : Calcul du Rendement du Portefeuille**

Pour chacun des $N$ scénarios générés, calculez le rendement global du portefeuille ($R_P$) en utilisant la formule :

$$
R_P = w_A R_A + w_B R_B
$$

In [14]:
R_portfolio = w_A * R[0] + w_B * R[1]

**Question 3 : Calcul de la Value at Risk (VaR)**


**Définition :** La VaR est la perte qui n'est dépassée que dans $1\%$ des cas. Elle correspond au centile à $1\%$ de la distribution des rendements $R_P$.

In [15]:
VaR_99_sim = np.percentile(R_portfolio, 1)
VaR_99_perte = -VaR_99_sim

**Question 4 : Comparaison avec la VaR Théorique**

Comparez la valeur de la VaR simulée (obtenue à la Question 3) avec la VaR théorique (analytique) que l'on peut calculer pour un portefeuille normal.



In [ ]:
mu_portfolio = w_A * mu_A + w_B * mu_B
var_portfolio = (w_A**2 * sigma_A**2 + w_B**2 * sigma_B**2 + 2 * w_A * w_B * rho * sigma_A * sigma_B)
sigma_portfolio = np.sqrt(var_portfolio)
z_01 = stats.norm.ppf(0.01) 
VaR_99_theorique = - (mu_portfolio + z_01 * sigma_portfolio)


**Question 5 : Question d'Analyse**

Pourquoi la méthode de Monte-Carlo serait-elle **indispensable** si les rendements ne suivaient pas une Loi Normale (par exemple, une loi de Student, plus réaliste en finance) ?

## 1. Limitations des approches analytiques

- Les formules théoriques supposent souvent des distributions normales  
- Les distributions réelles en finance ont des queues épaisses (*fat tails*)  
- Les rendements présentent souvent de l'asymétrie (*skewness* ≠ 0)  
- La kurtosis est généralement > 3 (événements extrêmes plus fréquents)

---

## 2. Avantages de la méthode Monte-Carlo

- **Flexibilité** : peut simuler n'importe quelle distribution  
- Permet d'utiliser des distributions réalistes (*Student-t*, *GARCH*, etc.)  
- Capture les dépendances non-linéaires entre actifs  
- Modélise l'hétéroscédasticité (volatilité changeante dans le temps)  
- Intègre des sauts (*jump processes*) et autres phénomènes complexes  

---

## 3. Démonstration avec la loi de Student

- **VaR normale à 99 %** : \( z = -2.33 \)  
- **VaR Student-t (ν = 5) à 99 %** : \( t = -3.36 \)  
- **Différence significative** : **+44 %** de risque extrême sous-estimé  

---

## 4. Vérification avec nos données simulées

- **Kurtosis simulée** : `kurtosis_simulee` (normale : 3.0) → queues légèrement plus épaisses  
- **Skewness simulée** : `skewness_simulee` (normale : 0.0) → légère asymétrie  
- Même sous hypothèse normale, Monte-Carlo valide l'approche analytique  

---

## 5. Conclusion

La méthode **Monte-Carlo** est indispensable car elle :

- Valide les approches analytiques sous hypothèses simples  
- Permet des modélisations réalistes sous hypothèses complexes  
- Fournit des intervalles de confiance pour les estimations  
- S'adapte facilement à de nouveaux modèles de risque  
